# Patient Feedback Sentiment Analysis

**Portfolio project: Healthcare Data Science**

This notebook builds a small end-to-end sentiment analysis pipeline for
patient satisfaction survey comments. It demonstrates two complementary
approaches:

1. **Lexicon-based sentiment scoring (VADER)** — fast, deterministic,
   fully local, no API costs.
2. **LLM-based structured extraction (Claude API)** — more nuanced,
   handles context/sarcasm better, and can return structured JSON
   (sentiment + aspects + a clinical safety flag) in a single call.

The dataset here is **synthetic** (fabricated for demonstration) so there
are no PHI/privacy concerns. In a real deployment this would be swapped
for de-identified survey exports (e.g. HCAHPS comments).


## 1. Setup

We use `pandas` for data handling, `vaderSentiment` for the lexicon-based
scoring, and `matplotlib`/`seaborn` for visualization.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 100)


## 2. Load the data

`patient_feedback.csv` contains 30 synthetic patient comments, each
tagged with a department. In practice this would come from a survey
export or EHR patient-experience module.


In [ ]:
df = pd.read_csv("patient_feedback.csv")
df.head(10)


In [ ]:
df.info()


## 3. Exploratory look at the text

Before scoring anything, it's worth checking comment length and
department distribution — this tells us whether any department is
under-represented, which would make its sentiment estimate less
reliable.


In [ ]:
df["word_count"] = df["comment"].str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["word_count"].plot(kind="hist", bins=8, ax=axes[0], color="#4C72B0")
axes[0].set_title("Comment length (words)")
axes[0].set_xlabel("Word count")

df["department"].value_counts().plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("Comments per department")

plt.tight_layout()
plt.show()


## 4. Sentiment scoring with VADER

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) is a
lexicon- and rule-based sentiment tool built for short, informal text.
Each word in its dictionary has a pre-scored valence (e.g. *"great"* is
strongly positive, *"terrible"* is strongly negative). VADER combines
these scores with rules for negation ("not good"), intensifiers ("very
good"), punctuation, and capitalization, then produces four scores per
piece of text:

| Score | Meaning |
|---|---|
| `neg` | proportion of text that is negative |
| `neu` | proportion of text that is neutral |
| `pos` | proportion of text that is positive |
| `compound` | a single normalized score from **-1** (most negative) to **+1** (most positive) — this is the one we'll use to classify overall sentiment |

VADER doesn't need training data or GPU/API calls, which makes it a
good, cheap first pass before reaching for an LLM.


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def score_sentiment(text):
    scores = analyzer.polarity_scores(text)
    return pd.Series(scores)

vader_scores = df["comment"].apply(score_sentiment)
df = pd.concat([df, vader_scores], axis=1)
df[["comment", "neg", "neu", "pos", "compound"]].head(10)


### 4.1 Turning the compound score into a label

VADER's own documentation suggests standard thresholds for the
compound score. We'll use the common convention:

- `compound >= 0.05` → **positive**
- `compound <= -0.05` → **negative**
- otherwise → **neutral**


In [ ]:
def label_sentiment(compound):
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["compound"].apply(label_sentiment)
df[["comment", "compound", "sentiment"]].head(10)


## 5. Visualizing the sentiment distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

order = ["positive", "neutral", "negative"]
colors = {"positive": "#55A868", "neutral": "#8C8C8C", "negative": "#C44E52"}

counts = df["sentiment"].value_counts().reindex(order)
axes[0].bar(counts.index, counts.values, color=[colors[s] for s in counts.index])
axes[0].set_title("Overall sentiment distribution")
axes[0].set_ylabel("Number of comments")

axes[1].pie(
    counts.values,
    labels=counts.index,
    autopct="%1.0f%%",
    colors=[colors[s] for s in counts.index],
    startangle=90,
)
axes[1].set_title("Sentiment share")

plt.tight_layout()
plt.show()


In [ ]:
sentiment_by_dept = pd.crosstab(df["department"], df["sentiment"])
sentiment_by_dept = sentiment_by_dept.reindex(columns=order, fill_value=0)

sentiment_by_dept.plot(
    kind="barh", stacked=True, figsize=(9, 5),
    color=[colors[s] for s in order]
)
plt.title("Sentiment by department")
plt.xlabel("Number of comments")
plt.legend(title="Sentiment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 6. Aspect tagging (rule-based)

A single sentiment label per comment hides *what* the patient is
actually reacting to. Aspect-based sentiment breaks comments down by
topic — e.g. wait time vs. staff communication vs. medication — so
that operational teams can act on the right lever.

Here we use a lightweight, transparent keyword-matching approach: each
aspect is defined by a small list of trigger words. This is easy to
audit and extend, though it will miss synonyms or indirect phrasing
that a more advanced model (like an LLM) would catch — that trade-off
is exactly why Section 8 shows the LLM-based alternative.


In [ ]:
aspect_keywords = {
    "wait_time": ["wait", "waited", "waiting", "delayed", "line", "minutes", "hours"],
    "staff_communication": ["explain", "explained", "listen", "heard", "rushed",
                             "dismissive", "informed", "follow up", "followed up",
                             "questions", "friendly", "kind", "attentive"],
    "medication": ["medication", "dosage", "prescription", "pharmacist", "side effects",
                   "pain management", "allergy"],
    "billing": ["bill", "billing", "overcharged", "itemized", "charge"],
    "facility": ["clean", "parking", "facility", "outdated"],
}

def tag_aspects(text):
    text_lower = text.lower()
    tags = [aspect for aspect, keywords in aspect_keywords.items()
            if any(kw in text_lower for kw in keywords)]
    return tags if tags else ["other"]

df["aspects"] = df["comment"].apply(tag_aspects)
df[["comment", "aspects"]].head(10)


In [ ]:
# Explode so each (comment, aspect) pair is its own row for aggregation
aspect_df = df.explode("aspects").reset_index(drop=True)

aspect_sentiment = pd.crosstab(aspect_df["aspects"], aspect_df["sentiment"])

aspect_sentiment = aspect_sentiment.reindex(columns=order, fill_value=0)

plt.figure(figsize=(8, 5))
sns.heatmap(aspect_sentiment, annot=True, fmt="d", cmap="RdYlGn_r", cbar=False)
plt.title("Sentiment counts by aspect")
plt.ylabel("Aspect")
plt.xlabel("Sentiment")
plt.tight_layout()
plt.show()


## 7. Clinical safety flagging

Beyond general sentiment, a healthcare feedback pipeline should surface
comments that hint at a **safety or care-quality issue** needing
follow-up — not just "unhappy customer" noise. Here we flag any
**negative** comment that also touches the `medication` aspect, since
medication-related complaints (wrong dosage, missed allergy check,
unexplained side effects) carry higher clinical risk than, say, a
parking complaint.

This is a simple rule for demonstration. In a production setting this
logic is exactly where clinical judgment (yours) adds real value over
a generic data science pipeline — e.g. weighting flags by department,
by severity keywords ("wrong", "error", "allergy"), or by repeat
patient history.


In [ ]:
def clinical_flag(row):
    return row["sentiment"] == "negative" and "medication" in row["aspects"]

df["clinical_flag"] = df.apply(clinical_flag, axis=1)

flagged = df[df["clinical_flag"]][["patient_id", "department", "comment", "compound"]]
flagged


In [ ]:
flagged.to_csv("flagged_for_review.csv", index=False)
print(f"{len(flagged)} comment(s) flagged for clinical review, saved to flagged_for_review.csv")


## 8. Alternative approach: LLM-based structured extraction (reference)

VADER is fast and free, but it scores words in isolation and can miss
context, sarcasm, or nuanced clinical language ("the patient is sick"
vs. "this deal is sick"). An LLM can read the whole sentence in context
and return **structured JSON** in one pass — sentiment, aspects, *and*
a reasoned clinical flag — instead of stitching together separate
rules.

The cell below is **not executed automatically** in this notebook
(it requires an `ANTHROPIC_API_KEY` environment variable and network
access to `api.anthropic.com`), but it shows the pattern for reference
and reuse.


In [ ]:
import os
import json
import anthropic

def analyze_comment_llm(comment, client=None):
    """Ask Claude to return structured sentiment + aspect + safety-flag JSON.

    Requires the ANTHROPIC_API_KEY environment variable to be set.
    """
    if client is None:
        client = anthropic.Anthropic()

    prompt = f"""Analyze this patient feedback comment. Return ONLY valid JSON
with these keys:
- "sentiment": one of positive, negative, neutral, mixed
- "aspects": list of topics mentioned, chosen from
  [wait_time, staff_communication, medication, billing, facility, other]
- "clinical_flag": true if this suggests a safety or care-quality issue
  needing follow-up, else false
- "rationale": one-sentence explanation of the clinical_flag decision

Comment: \"{comment}\""""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    return json.loads(response.content[0].text)

# Example usage (uncomment to run with a valid API key set in your environment):
# result = analyze_comment_llm(
#     "Nurse administered medication without confirming my allergy history."
# )
# print(result)


## 9. Summary

- **VADER** gave us a fast, transparent, zero-cost first pass across
  all 30 comments, with an overall sentiment split and a breakdown by
  department.
- **Rule-based aspect tagging** let us see *what* patients were
  reacting to (wait time, staff communication, medication, billing,
  facility) rather than just a single positive/negative label.
- **Clinical flagging** combined sentiment + aspect to surface the
  comments most worth a human follow-up — the kind of triage logic
  that turns a generic NLP exercise into something a patient
  experience or quality team could actually use.
- The **LLM-based approach** (Section 8) is the natural next step for
  production use: better handling of context and sarcasm, and richer
  structured output, at the cost of an API call per comment.

### Possible extensions
- Swap in a real de-identified survey export (e.g. HCAHPS comments).
- Compare VADER labels against the LLM labels on the same comments to
  quantify where the simple lexicon approach disagrees with the
  context-aware one.
- Track sentiment trend over time (weekly/monthly) once comments carry
  a date field.
- Add a lightweight Streamlit or Power BI dashboard on top of
  `flagged_for_review.csv` for a patient experience team to triage.
